# Exploratory Data Analysis (EDA)

This notebook explores the dataset to uncover patterns, identify data quality issues, and form hypotheses to guide feature engineering.

**Objective:**
- Understand data structure and quality
- Identify missing values and outliers
- Discover patterns and relationships
- Generate insights for feature engineering

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from scipy import stats

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)

## 1. Overview of the Data

Understanding the structure of the dataset.

In [ ]:
df = pd.read_csv('../data/data.csv')
print('Dataset Shape:', df.shape)
print('\nData Types:')
print(df.dtypes)
df.info()
df.head(10)

## 2. Summary Statistics

Understanding central tendency and dispersion.

In [ ]:
print('Numerical Features Summary:')
print(df.describe().T)
print('\nSkewness:')
numerical_cols = df.select_dtypes(include=[np.number]).columns
for col in numerical_cols:
    print(f'{col}: {df[col].skew():.4f}')

## 3. Distribution of Numerical Features

Visualize distributions of numerical features.

In [ ]:
numerical_cols = df.select_dtypes(include=[np.number]).columns
n_rows = (len(numerical_cols) + 2) // 3
fig, axes = plt.subplots(n_rows, 3, figsize=(15, 4*n_rows))
axes = axes.flatten()
for idx, col in enumerate(numerical_cols):
    axes[idx].hist(df[col].dropna(), bins=30, color='skyblue')
    axes[idx].set_title(f'Distribution of {col}')
for idx in range(len(numerical_cols), len(axes)):
    axes[idx].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(n_rows, 3, figsize=(15, 4*n_rows))
axes = axes.flatten()
for idx, col in enumerate(numerical_cols):
    df[col].dropna().plot(kind='kde', ax=axes[idx], color='steelblue')
    axes[idx].set_title(f'KDE of {col}')
for idx in range(len(numerical_cols), len(axes)):
    axes[idx].set_visible(False)
plt.tight_layout()
plt.show()

## 4. Distribution of Categorical Features

Analyze categorical feature distributions.

In [ ]:
categorical_cols = df.select_dtypes(include='object').columns
n_rows = (len(categorical_cols) + 1) // 2
fig, axes = plt.subplots(n_rows, 2, figsize=(15, 4*n_rows))
axes = axes.flatten() if n_rows > 1 else axes
for idx, col in enumerate(categorical_cols):
    df[col].value_counts().plot(kind='bar', ax=axes[idx], color='coral')
    axes[idx].set_title(f'Distribution of {col}')
for idx in range(len(categorical_cols), len(axes)):
    axes[idx].set_visible(False)
plt.tight_layout()
plt.show()

## 5. Correlation Analysis

Identify relationships between numerical features.

In [ ]:
correlation_matrix = df[numerical_cols].corr()
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()
print('\nHighly Correlated Pairs (> 0.7):')
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > 0.7:
            print(f'{correlation_matrix.columns[i]} - {correlation_matrix.columns[j]}: {correlation_matrix.iloc[i, j]:.4f}')

## 6. Identifying Missing Values

Detect and visualize missing data patterns.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
print('Missing Values:')
print(pd.DataFrame({'Count': missing, 'Percentage': missing_pct})[missing > 0])
if missing.sum() > 0:
    plt.figure(figsize=(12, 6))
    missing[missing > 0].plot(kind='bar', color='crimson')
    plt.title('Missing Values by Column')
    plt.tight_layout()
    plt.show()

## 7. Outlier Detection

Identify outliers using box plots and statistical methods.

In [ ]:
fig, axes = plt.subplots(n_rows, 3, figsize=(15, 4*n_rows))
axes = axes.flatten()
for idx, col in enumerate(numerical_cols):
    axes[idx].boxplot(df[col].dropna())
    axes[idx].set_title(f'Box Plot of {col}')
for idx in range(len(numerical_cols), len(axes)):
    axes[idx].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
print('Outlier Detection (IQR Method):')
for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f'{col}: {len(outliers)} outliers ({len(outliers)/len(df)*100:.2f}%)')

In [ ]:
print('Outlier Detection (Z-score Method, |z| > 3):')
for col in numerical_cols:
    z_scores = np.abs(stats.zscore(df[col].dropna()))
    outliers = np.sum(z_scores > 3)
    print(f'{col}: {outliers} outliers ({outliers/len(df)*100:.2f}%)')
print('\nEDA Analysis Complete!')